# Stage 2: Activation Hooking and HybridCBM Optimization

This notebook demonstrates how to load a model (compatible with Kaggle environments), register a forward hook to extract Layer 14 intermediate activations, cache them to `.safetensors`, and train the `HybridCBM` layer to reconstruct the activations using a combination of static concepts and learnable dynamic concepts.

## 1. Install Dependencies & Setup Environment

In [ ]:
# Install core requirements
!pip install -q lightning pytorch-lightning transformers accelerate safetensors scikit-learn

In [ ]:
import os
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

# Add repo src to path if needed
import sys
sys.path.append(os.path.abspath('..'))

## 2. Mock Model Setup (For demonstration or local execution)

We define a tiny mock model structure resembling a transformer to verify the hooking extraction pipeline without downloading a large model, then show how to load the real model.

In [ ]:
class MockLayer(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.linear = nn.Linear(d_model, d_model)
    def forward(self, x):
        return self.linear(x)

class MockModel(nn.Module):
    def __init__(self, d_model=1536, n_layers=20):
        super().__init__()
        self.layers = nn.ModuleList([MockLayer(d_model) for _ in range(n_layers)])
        self.model = nn.Module() # Mock the model.model nest
        self.model.layers = self.layers
        
    def forward(self, x):
        curr = x
        for layer in self.layers:
            curr = layer(curr)
        return curr

print("Initializing Mock Model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MockModel(d_model=1536).to(device)
print(model)

## 3. Activation Hooking & Extraction

We use `ActivationHookExtractor` to register a forward hook on Layer 14.

In [ ]:
from src.system1.hook_extractor import ActivationHookExtractor

# Layer path for our mock transformer layers
layer_path = "model.layers[14]"

extractor = ActivationHookExtractor(model, layer_path=layer_path)
extractor.register()

# Run mock inputs (simulating a batch size of 8 prompts, sequence length 32)
mock_inputs = torch.randn(8, 32, 1536).to(device)

with torch.no_grad():
    _ = model(mock_inputs)

print(f"Extracted activation blocks: {len(extractor.extracted_activations)}")
print(f"Shape of first block: {extractor.extracted_activations[0].shape}")

## 4. Cache Activations to Safetensors

We save the captured tensors as `.safetensors` on disk.

In [ ]:
cache_path = "./cached_activations/layer_14_activations.safetensors"
extractor.save_cache(cache_path)
extractor.remove()
extractor.clear()

## 5. Load Cached Tensors & HybridCBM Layer Optimization

Load the activations from disk and train the HybridCBM projections to minimize the reconstruction loss.

In [ ]:
from safetensors.torch import load_file
from src.system1.hybrid_cbm import HybridCBM

# Load cached activations
cached_data = load_file(cache_path)
activations = cached_data["activations"].to(device)
print(f"Loaded activations shape: {activations.shape}")

# Initialize HybridCBM with dynamically inferred input shape, 
# 10 default concepts, and 5 dynamic learnable dimensions
    # Initialize HybridCBM with explicit emb_dim to prevent empty parameter list errors
    hybrid_cbm = HybridCBM(n_dynamic=5, clip_dim=512, emb_dim=activations.shape[-1]).to(device)

# Setup Optimizer targeting projection and decoder weights
optimizer = torch.optim.Adam(hybrid_cbm.parameters(), lr=0.01)

print("Training HybridCBM representation decomposition...")
for epoch in range(100):
    optimizer.zero_grad()
    
    # Forward pass: projects activations, decodes back, and calculates reconstruction loss
    z, x_rec, rec_loss = hybrid_cbm(activations)
    
    rec_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:03d} | Reconstruction Loss (MSE): {rec_loss.item():.6f}")

## 6. Inspecting the Bottleneck States

We can check the resulting hybrid bottleneck dimension ($z$):

In [ ]:
print(f"Concept Bottleneck Tensor shape (Batch, SeqLen, Concepts): {z.shape}")
print(f"Total concepts: {hybrid_cbm.n_static} static + {hybrid_cbm.n_dynamic} dynamic = {z.shape[-1]}")
print("Static Concepts list:", hybrid_cbm.concepts)

## 7. Stage 3: Incremental Concept Translation

After training, project the learned dynamic concepts into the Candidate Concept Bank via CLIP space cosine similarity to assign human-understandable labels.

In [ ]:
# Define a candidate concept bank
candidate_labels = [
    "spatial puzzle solving",
    "matrix grid rotation",
    "pattern matching",
    "iterative loop syntax",
    "negation operator",
    "recursive backtracking",
    "binary arithmetic",
    "logical conjunction",
    "wikidata mapping"
]

# Mock precomputed normalized CLIP text embeddings for candidate concepts
candidate_embeddings = torch.randn(len(candidate_labels), 512).to(device)
candidate_embeddings = torch.nn.functional.normalize(candidate_embeddings, p=2, dim=-1)

# Perform translation
translated_labels, similarities = hybrid_cbm.translate_dynamic_concepts(candidate_embeddings, candidate_labels)

print("Translated Dynamic Concepts:")
for i, (label, sim) in enumerate(zip(translated_labels, similarities)):
    print(f"  Dynamic Concept {i+1} -> Label: '{label}' (Similarity: {sim.item():.4f})")